# Claude with Google Vertex AI - Prompt Evaluation

Description of the lesson.


In [ ]:
import sys
sys.path.insert(0, "..")

from utils import setup_client, add_user_message, add_assistant_message, chat

client, model = setup_client()

In [ ]:
import json

def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts 
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of objects, 
    each representing task that requires Python, JSON, or a Regex to complete.
    
    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": "json" or "python" or "regex"
            "solution_criteria": "Key criteria for evaluating the solution"
        },
        ...additional
    ]
    ```
    
    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code
    
    Please generate 3 objects.
    """

    messages = []

    add_user_message(
        messages,
        prompt,
    )

    add_assistant_message(
        messages,
        "```json"
    )

    text = chat(client, model, messages, stop_sequences=["```"])

    return json.loads(text)


In [ ]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
        Please solve the following task:

        {test_case["task"]}

        * Respond only with Python, JSON, or a plain Regex
        * Do not add any comments or commentary or explanation
        """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(client, model, messages, stop_sequences=["```"])

    return output

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
        You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

        Original Task:
        <task>
        {test_case["task"]}
        </task>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {test_case["solution_criteria"]}
        </criteria>

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """

    messages = []

    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(client, model, messages, stop_sequences=["```"])

    return json.loads(eval_text)

In [ ]:
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)

    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

# print(json.dumps(results, indent=2))

In [ ]:
from IPython.display import display, Markdown

for i, result in enumerate(results, 1):
    task = result["test_case"]["task"]
    score = result["score"]
    reasoning = result["reasoning"]
    output = result["output"]

    display(Markdown(f"""
---
### Test Case {i}

**Task:** {task}

**Score:** {score}/10

**Reasoning:** {reasoning}

**Output:**

{output}
"""))